# XVA Aggregation — CVA + DVA + FVA + MVA under a Netting Set
## QRE-56

This notebook assembles the full XVA stack from QRE-53–55 under a **netting set** — a group of trades governed by a single ISDA Master Agreement. The key new element is **netting**: exposure is computed on the **net portfolio MTM**, not trade-by-trade, which materially reduces all XVA components.

$$V^{\text{risky}} = V^{\text{risk-free}} - \text{CVA} + \text{DVA} - \text{FVA} - \text{MVA}$$

**What changes from the single-trade notebooks:**

| Component | Single trade | Netting set |
|---|---|---|
| MTM | $V(t)$ per trade | $V_{\text{net}}(t) = \sum_k V_k(t)$ |
| EE profile | per trade | on net portfolio → lower |
| CVA | $\sum_k \text{CVA}_k$ (over-estimates) | CVA(net portfolio) ≤ $\sum_k \text{CVA}_k$ |
| DVA | per trade | on net NEE |
| FVA | $\sum_k \text{FVA}_k$ | on net EE/NEE → netting benefit |
| MVA (SIMM) | per trade DV01 | net DV01 (offsetting trades reduce IM) |

The **netting set** in this notebook: a 5Y payer IRS (QRE-53 reference trade) partially hedged by a 3Y receiver IRS with half the notional.


In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quant_risk.setup import base
from quant_risk.config import PROCESSED_DIR
from quant_risk.curves.ois import OISCurve
from quant_risk.models.rates import HullWhiteProcess
from quant_risk.models.simulator import MCSimulator

np, pd, plt = base()

RNG_SEED = 42


---
## 1. Netting Sets and Exposure Reduction

### 1.1 How Netting Reduces Exposure

Under a bilateral ISDA Master Agreement with a netting clause, upon default the outstanding trades are **closed out at their current MTM and netted**. Only the net position is owed. This means:

$$\text{EE}_{\text{net}}(t) = \mathbb{E}\bigl[\max\bigl(\textstyle\sum_k V_k(t),\, 0\bigr)\bigr] \leq \sum_k \mathbb{E}\bigl[\max(V_k(t), 0)\bigr] = \sum_k \text{EE}_k(t)$$

The inequality is strict whenever there exist paths where positive and negative trade MTMs partially offset. The **netting benefit** is:

$$\text{Netting benefit} = \sum_k \text{EE}_k(t) - \text{EE}_{\text{net}}(t) \geq 0$$

### 1.2 Netting Ratio

The netting ratio $\rho_{\text{net}}$ measures how much the netting set compresses exposure:

$$\rho_{\text{net}} = \frac{\text{EE}_{\text{net}}(t)}{\sum_k \text{EE}_k(t)} \in [0, 1]$$

$\rho_{\text{net}} = 1$: perfectly correlated trades, no netting benefit. $\rho_{\text{net}} \to 0$: perfectly offsetting trades, exposure collapses to near zero.

### 1.3 Close-Out Netting vs Payment Netting

**Close-out netting** (on default): applies to all outstanding trades under the ISDA Master Agreement — this is what drives the XVA reduction.

**Payment netting** (ongoing): cash flows on the same date in the same currency are netted into a single payment — reduces settlement risk but does not affect XVA.

### 1.4 SIMM Netting: Sensitivity Aggregation

For MVA, SIMM aggregates **net sensitivities** across trades within a risk class. For two interest rate swaps in the same currency bucket:

$$\text{IM}_{\text{net}} = \text{RW} \times |\text{DV01}_1 + \text{DV01}_2|$$

Since a payer swap has positive DV01 and a receiver has negative DV01, a partial hedge in the same currency dramatically reduces IM — and hence MVA. This is SIMM's primary netting mechanism.


In [ ]:
# ── Load OIS curve and build HW simulator ─────────────────────────────────────
try:
    ois = OISCurve.from_processed(str(PROCESSED_DIR))
    print(ois.describe())
except FileNotFoundError:
    print("Using synthetic OIS curve")
    data = pd.DataFrame(
        {"years":           [1/12,2/12,3/12,6/12,9/12,1.0,2.0,3.0,5.0,10.0,15.0],
         "zero_rate_pct":   [2.63,2.61,2.58,2.48,2.38,2.30,2.20,2.15,2.20,2.40,2.50],
         "discount_factor": [np.exp(-r/100*t) for r,t in zip(
             [2.63,2.61,2.58,2.48,2.38,2.30,2.20,2.15,2.20,2.40,2.50],
             [1/12,2/12,3/12,6/12,9/12,1,2,3,5,10,15])],
         "valuation_date":  ["2026-03-24"]*11},
        index=["1M","2M","3M","6M","9M","12M","2Y","3Y","5Y","10Y","10Y+"])
    data.index.name = "maturity"
    ois = OISCurve(data)

KAPPA, SIGMA = 0.10, 0.50
r0 = ois.forward_rate(1/12, 2/12)

sim = MCSimulator(
    process   = HullWhiteProcess(curve=ois, kappa=KAPPA, sigma=SIGMA),
    x0        = r0,
    T         = 10.0,
    n_steps   = 120,
    n_paths   = 5000,
    antithetic= True,
    seed      = RNG_SEED,
)
print(sim.describe())

# ── ATM rates for each tenor ───────────────────────────────────────────────────
def par_swap_rate(maturity: float, curve: OISCurve) -> float:
    """Par swap rate for an annual-pay IRS of given maturity (in percent)."""
    dates   = np.arange(1.0, maturity + 0.001, 1.0)
    dfs     = np.array([curve.discount_factor(T) for T in dates])
    annuity = dfs.sum()
    return (1.0 - dfs[-1]) / annuity * 100

K_5Y = par_swap_rate(5.0, ois)
K_3Y = par_swap_rate(3.0, ois)
print(f"\nATM par swap rates:")
print(f"  3Y: {K_3Y:.4f}%")
print(f"  5Y: {K_5Y:.4f}%")


In [ ]:
# ── HW bond price, IRS MTM, DV01 — all parameters as arguments ───────────────

def hw_B(tau, kappa):
    return np.where(tau > 0, (1 - np.exp(-kappa*tau))/kappa, 0.0)

def hw_bond_price_paths(t, maturities, r_t, kappa, sigma, curve):
    tau     = maturities - t
    B_tau   = hw_B(tau, kappa)
    P_0T    = np.array([curve.discount_factor(T) for T in maturities])
    P_0t    = curve.discount_factor(t) if t > 1e-6 else 1.0
    dt_fd   = 1/12; t_lo = max(t - dt_fd, dt_fd)
    f_0t    = curve.forward_rate(t_lo, t_lo + dt_fd)
    sigma_d = sigma/100
    var_adj = (sigma_d**2/(4*kappa))*B_tau**2*(1-np.exp(-2*kappa*t))
    rate_dev = (r_t[:,None]-f_0t)/100*B_tau[None,:]
    return (P_0T/P_0t)[None,:]*np.exp(-rate_dev-var_adj[None,:])

def irs_mtm(paths, t, payment_dates, year_fracs,
             K, notional, is_payer, kappa, sigma, curve, dt, n_steps):
    remaining = payment_dates[payment_dates > t]
    yf_r      = year_fracs[payment_dates > t]
    if len(remaining) == 0:
        return np.zeros(paths.shape[0])
    t_idx  = min(int(round(t/dt)), n_steps)
    r_t    = paths[:, t_idx]
    P_ti   = hw_bond_price_paths(t, remaining, r_t, kappa, sigma, curve)
    fl     = 1.0 - P_ti[:,-1]
    fx     = (K/100)*(yf_r*P_ti).sum(axis=1)
    return notional*(fl-fx) if is_payer else -notional*(fl-fx)

def irs_dv01(paths, t, payment_dates, year_fracs,
              notional, kappa, sigma, curve, dt, n_steps):
    remaining = payment_dates[payment_dates > t]
    yf_r      = year_fracs[payment_dates > t]
    if len(remaining) == 0:
        return np.zeros(paths.shape[0])
    t_idx  = min(int(round(t/dt)), n_steps)
    r_t    = paths[:, t_idx]
    P_ti   = hw_bond_price_paths(t, remaining, r_t, kappa, sigma, curve)
    annuity = (yf_r*P_ti).sum(axis=1)
    return notional/10_000 * annuity

# ── CVA, FVA, MVA functions (carry over from QRE-53/54/55) ───────────────────
def hazard_rate(cds_bps, recovery):
    return (cds_bps/10_000)/(1-recovery)

def compute_cva(profile, cds_bps, recovery):
    h     = hazard_rate(cds_bps, recovery)
    dates = profile['dates']
    t_prev = np.concatenate([[0.0], dates[:-1]])
    pd_per = np.exp(-h*t_prev) - np.exp(-h*dates)
    cva_bp = (1-recovery)*profile['EE_disc']*pd_per
    return {'CVA': float(cva_bp.sum()), 'cva_by_period': cva_bp}

def compute_dva(profile, nee_disc, own_cds_bps, own_recovery, sim, dates):
    h_own = hazard_rate(own_cds_bps, own_recovery)
    t_prev = np.concatenate([[0.0], dates[:-1]])
    pd_own = np.exp(-h_own*t_prev) - np.exp(-h_own*dates)
    dva_bp = (1-own_recovery)*np.abs(nee_disc)*pd_own
    return {'DVA': float(dva_bp.sum()), 'dva_by_period': dva_bp}

def compute_fva(ee_disc, nee_disc, dates, s_f_bps, s_r_bps, csa_type):
    s_f = s_f_bps/10_000; s_r = s_r_bps/10_000
    t_prev = np.concatenate([[0.0], dates[:-1]]); dt_i = dates - t_prev
    if csa_type == 'two_way':
        return {'FVA':0.,'FCA':0.,'FBA':0.,
                'fca_by_period':np.zeros_like(ee_disc),
                'fba_by_period':np.zeros_like(ee_disc)}
    fca_p = s_f*dt_i*ee_disc
    fba_p = s_r*dt_i*np.abs(nee_disc) if csa_type != 'one_way_post' else np.zeros_like(ee_disc)
    fca=float(fca_p.sum()); fba=float(fba_p.sum())
    return {'FVA':fca-fba,'FCA':fca,'FBA':fba,'fca_by_period':fca_p,'fba_by_period':fba_p}

def compute_mva_from_dv01(paths, exp_dates, sim, payment_dates_list,
                            year_fracs_list, notionals, kappa, sigma, curve,
                            rw_bps, mpor_days, s_im_bps):
    """Net SIMM MVA across a list of trades — DV01s sum within the same IR bucket."""
    s_im  = s_im_bps/10_000
    rw    = rw_bps/10_000
    mpor_adj = np.sqrt(mpor_days/10)
    t_prev = np.concatenate([[0.0], exp_dates[:-1]]); dt_i = exp_dates - t_prev
    eim_disc = np.zeros(len(exp_dates))
    for j, t in enumerate(exp_dates):
        # Sum DV01 across all trades in the netting set (SIMM intra-bucket netting)
        net_dv01 = sum(
            irs_dv01(paths, t, pd_i, yf_i, n_i, kappa, sigma, curve, sim.dt, sim.n_steps)
            for pd_i, yf_i, n_i in zip(payment_dates_list, year_fracs_list, notionals)
        )
        im_net  = rw * np.abs(net_dv01) * mpor_adj
        eim_disc[j] = (sim.sdf(t) * im_net).mean()
    mva_p = s_im * dt_i * eim_disc
    return {'MVA': float(mva_p.sum()), 'eim_disc': eim_disc, 'mva_by_period': mva_p}


In [ ]:
# ── Netting set definition — two trades ──────────────────────────────────────
#
# Trade 1: 5Y payer IRS, K=ATM, N=1M EUR  (pay fixed, receive floating)
# Trade 2: 3Y receiver IRS, K=ATM, N=500K EUR  (receive fixed, partial hedge)
#
# The receiver offsets a portion of the payer's interest rate risk.
# Net DV01 ~ DV01_payer_5Y - DV01_receiver_3Y (opposing signs)

trades = [
    dict(label='Trade 1: 5Y Payer IRS',
         payment_dates = np.arange(1.0, 5.001, 1.0),
         year_fracs    = np.ones(5),
         K             = K_5Y,
         notional      = 1_000_000,
         is_payer      = True),
    dict(label='Trade 2: 3Y Receiver IRS',
         payment_dates = np.arange(1.0, 3.001, 1.0),
         year_fracs    = np.ones(3),
         K             = K_3Y,
         notional      =   500_000,
         is_payer      = False),
]

exp_dates = np.arange(0.5, 5.5, 0.5)   # semi-annual, out to 5Y (longest trade)

# Build exposure profiles: individual and netting-set
profiles_individual = []
for tr in trades:
    def mtm_tr(paths, t, tr=tr):
        return irs_mtm(paths, t,
                       tr['payment_dates'], tr['year_fracs'],
                       tr['K'], tr['notional'], tr['is_payer'],
                       KAPPA, SIGMA, ois, sim.dt, sim.n_steps)
    prof = sim.exposure_profile(mtm_tr, exp_dates)
    profiles_individual.append(prof)
    print(f"{tr['label']:35s}  "
          f"EE_peak={prof['EE'].max():>10,.0f}  "
          f"EPE={prof['EPE']:>10,.0f}  "
          f"NEE_min={prof['NEE'].min():>10,.0f}")

# Netting set: MTM is the sum of all trades
def mtm_net(paths, t):
    return sum(
        irs_mtm(paths, t,
                tr['payment_dates'], tr['year_fracs'],
                tr['K'], tr['notional'], tr['is_payer'],
                KAPPA, SIGMA, ois, sim.dt, sim.n_steps)
        for tr in trades
    )

profile_net = sim.exposure_profile(mtm_net, exp_dates)
disc_matrix = np.column_stack([sim.sdf(t) for t in exp_dates])
nee_disc_net = (disc_matrix * np.minimum(profile_net['mtm'], 0)).mean(axis=0)

gross_ee = sum(p['EE'] for p in profiles_individual)
netting_benefit = gross_ee - profile_net['EE']
netting_ratio   = profile_net['EE'] / np.where(gross_ee > 0, gross_ee, 1.0)

print(f"\nNetting set summary:")
print(f"  Gross EE peak (sum):  {gross_ee.max():>10,.0f} EUR")
print(f"  Net   EE peak:        {profile_net['EE'].max():>10,.0f} EUR")
print(f"  Max netting benefit:  {netting_benefit.max():>10,.0f} EUR")
print(f"  Min netting ratio:    {netting_ratio[gross_ee > 0].min():.3f}  (1.0 = no benefit)")


In [ ]:
# ── Exposure profile: individual trades vs netting set ───────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: EE comparison
ax = axes[0]
colors_ind = ['steelblue', 'darkorange']
for i, (tr, prof) in enumerate(zip(trades, profiles_individual)):
    ax.plot(exp_dates, prof['EE']/1e4, '--', color=colors_ind[i], lw=1.5,
            alpha=0.8, label=f'{tr["label"]} EE (gross)')
ax.fill_between(exp_dates, 0, profile_net['EE']/1e4,
                alpha=0.20, color='purple')
ax.plot(exp_dates, profile_net['EE']/1e4, '-', color='purple', lw=2.5,
        label='Netting set EE (net)')
ax.plot(exp_dates, gross_ee/1e4, ':', color='black', lw=1.5, alpha=0.7,
        label='Gross EE (sum of trades)')
ax.fill_between(exp_dates, profile_net['EE']/1e4, gross_ee/1e4,
                alpha=0.15, color='green', label='Netting benefit')
ax.set_xlabel('t (years)')
ax.set_ylabel('Expected Exposure (EUR × 10⁴)')
ax.set_title('Netting benefit: gross vs net EE profile')
ax.legend(fontsize=7)

# Right: full exposure fan for netting set
ax = axes[1]
ax.fill_between(exp_dates, 0, profile_net['EE']/1e4,
                alpha=0.25, color='purple', label='EE_net')
ax.plot(exp_dates, profile_net['EE']  /1e4, '-',  color='purple',  lw=2.0)
ax.plot(exp_dates, profile_net['PFE'] /1e4, '--', color='darkorange', lw=1.8,
        label='PFE 95th pctile')
ax.fill_between(exp_dates, profile_net['NEE']/1e4, 0,
                alpha=0.20, color='firebrick', label='NEE_net')
ax.plot(exp_dates, profile_net['NEE']/1e4,  '-',  color='firebrick', lw=1.5)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('t (years)')
ax.set_ylabel('Net exposure (EUR × 10⁴)')
ax.set_title('Netting set exposure profile\n(5Y payer + 3Y receiver partial hedge)')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

print(f"Netting ratios by date:")
for t, nr in zip(exp_dates, netting_ratio):
    print(f"  t={t:.1f}Y  NR={nr:.3f}")


---
## 2. Full XVA Aggregation

### 2.1 Aggregation Under the Netting Set

Each XVA component uses the **net portfolio** exposure profiles, not per-trade profiles. The comparison between gross (sum-of-trades) and net (netting set) quantifies the value of the ISDA Master Agreement.

### 2.2 BCVA = CVA − DVA

Under bilateral valuation (IFRS 13):

$$\text{BCVA} = \text{CVA} - \text{DVA}$$

- **CVA**: risk of counterparty default while we're in-the-money
- **DVA**: benefit from own default while we're out-of-the-money
- Both use the same discounted exposure profiles (EE_disc and |NEE_disc|)

The sign convention: CVA is a **deduction** from risk-free MtM; DVA is an **addition** (own-default benefit). BCVA is the net bilateral adjustment.

### 2.3 XVA Reserve

Under IFRS 13, the total fair value adjustment is:

$$V^{\text{fair value}} = V^{\text{risk-free}} - \text{CVA} + \text{DVA} - \text{FVA} - \text{MVA}$$

The XVA desk books this as a day-one reserve. Subsequent P&L comes from:
- Changes in exposure profiles (rate moves → MTM changes)
- Changes in credit spreads (CS01)
- Passage of time (theta decay of the XVA)


In [ ]:
# ── Full XVA aggregation — all parameters as arguments ────────────────────────

# Counterparty parameters
CDS_CPTY   = 60.0    # bps — A-rated counterparty
RECOVERY   = 0.40

# Own-credit parameters
CDS_OWN    = 45.0    # bps — our own CDS
OWN_RECOV  = 0.40

# Funding parameters
S_FUND     = 30.0    # bps — unsecured funding above OIS
S_INVEST   = 20.0    # bps — reinvestment spread above OIS
CSA_TYPE   = 'none'  # no CSA on this netting set

# IM parameters
SIMM_RW_BPS   = 64.0
MPOR_DAYS     = 10
IM_FUND_SPREAD = 25.0  # bps

# ── CVA ───────────────────────────────────────────────────────────────────────
cva_net  = compute_cva(profile_net, CDS_CPTY, RECOVERY)
cva_gross = {
    'CVA': sum(compute_cva(p, CDS_CPTY, RECOVERY)['CVA'] for p in profiles_individual)
}

# ── DVA ───────────────────────────────────────────────────────────────────────
dva_net = compute_dva(profile_net, nee_disc_net, CDS_OWN, OWN_RECOV, sim, exp_dates)

# ── FVA ───────────────────────────────────────────────────────────────────────
fva_net   = compute_fva(profile_net['EE_disc'], nee_disc_net, exp_dates,
                          S_FUND, S_INVEST, CSA_TYPE)

# Gross FVA (sum over individual trades, each with own nee_disc)
fva_gross_total = 0.0
for tr, prof in zip(trades, profiles_individual):
    disc_tr = np.column_stack([sim.sdf(t) for t in exp_dates])
    nee_tr  = (disc_tr * np.minimum(prof['mtm'], 0)).mean(axis=0)
    fva_tr  = compute_fva(prof['EE_disc'], nee_tr, exp_dates, S_FUND, S_INVEST, CSA_TYPE)
    fva_gross_total += fva_tr['FVA']

# ── MVA ───────────────────────────────────────────────────────────────────────
pd_list = [tr['payment_dates'] for tr in trades]
yf_list = [tr['year_fracs']    for tr in trades]
no_list = [tr['notional']      for tr in trades]

mva_net  = compute_mva_from_dv01(
    sim.paths, exp_dates, sim,
    pd_list, yf_list, no_list,
    KAPPA, SIGMA, ois,
    SIMM_RW_BPS, MPOR_DAYS, IM_FUND_SPREAD
)

# Gross MVA (each trade independently)
mva_gross_total = 0.0
for tr in trades:
    mva_tr = compute_mva_from_dv01(
        sim.paths, exp_dates, sim,
        [tr['payment_dates']], [tr['year_fracs']], [tr['notional']],
        KAPPA, SIGMA, ois, SIMM_RW_BPS, MPOR_DAYS, IM_FUND_SPREAD
    )
    mva_gross_total += mva_tr['MVA']

# ── Summary table ──────────────────────────────────────────────────────────────
total_notional = sum(tr['notional'] for tr in trades)
bcva_net   = cva_net['CVA'] - dva_net['DVA']
xva_net    = cva_net['CVA'] - dva_net['DVA'] + fva_net['FVA'] + mva_net['MVA']
xva_gross  = cva_gross['CVA'] + fva_gross_total + mva_gross_total

print("=" * 72)
print(f"XVA Aggregation — 2-Trade Netting Set  (total notional EUR {total_notional/1e6:.1f}M)")
print("=" * 72)
print(f"{'Component':<12}  {'Gross (no netting)':>20}  {'Net (netting set)':>18}  {'Netting benefit':>16}")
print("-" * 72)
for label, gross, net in [
    ('CVA',  cva_gross['CVA'],   cva_net['CVA']),
    ('DVA',  0.0,                dva_net['DVA']),
    ('FVA',  fva_gross_total,    fva_net['FVA']),
    ('MVA',  mva_gross_total,    mva_net['MVA']),
]:
    benefit = gross - net if gross > net else 0.0
    print(f"  {label:<10}  {gross:>20,.2f}  {net:>18,.2f}  {benefit:>16,.2f}")
print("-" * 72)
print(f"  {'BCVA':<10}  {'':>20}  {bcva_net:>18,.2f}")
print(f"  {'Total XVA':<10}  {xva_gross:>20,.2f}  {xva_net:>18,.2f}  "
      f"{xva_gross - xva_net:>16,.2f}")
print("=" * 72)
print(f"  Total XVA in bps of total notional: "
      f"Gross={xva_gross/total_notional*10000:.2f}  "
      f"Net={xva_net/total_notional*10000:.2f}")
print(f"  Netting saves: {(xva_gross-xva_net)/total_notional*10000:.2f} bps "
      f"= {(xva_gross-xva_net)/xva_gross*100:.1f}% of gross XVA")


In [ ]:
# ── XVA waterfall chart: gross vs net ────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: gross vs net XVA by component
components = ['CVA', 'DVA', 'FVA', 'MVA']
gross_vals = [cva_gross['CVA'], 0.0, fva_gross_total, mva_gross_total]
net_vals   = [cva_net['CVA'],   dva_net['DVA'], fva_net['FVA'], mva_net['MVA']]
colors_xva = ['firebrick', 'steelblue', 'darkorange', 'purple']

ax = axes[0]
x = np.arange(len(components))
w = 0.30
bars_g = ax.bar(x - w/2, [v/SWAP_NOTIONAL*10000 for v in gross_vals],
                 width=w, alpha=0.4, color=colors_xva, label='Gross (no netting)')
bars_n = ax.bar(x + w/2, [v/SWAP_NOTIONAL*10000 for v in net_vals],
                 width=w, alpha=0.9, color=colors_xva, label='Net (netting set)')
SWAP_NOTIONAL = 1_000_000  # reference notional for bps scaling
ax.set_xticks(x); ax.set_xticklabels(components)
ax.set_ylabel('Adjustment (bps of 1M notional)')
ax.set_title('XVA by component: gross vs net')
ax.legend(fontsize=7)
for bar, v in zip(bars_n, net_vals):
    ax.annotate(f'{v/SWAP_NOTIONAL*10000:.1f}',
                xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                xytext=(0,4), textcoords='offset points', ha='center', fontsize=8)

# Right: XVA waterfall — MtM → XVA components → risky MtM
ax = axes[1]
# Waterfall items: CVA(cost), +DVA(benefit), -FVA(cost), -MVA(cost)
items  = ['Risk-free MtM', 'CVA
(cost)', 'DVA
(benefit)', 'FVA
(cost)', 'MVA
(cost)', 'Fair Value']
values = [0.0, -cva_net['CVA'], dva_net['DVA'], -fva_net['FVA'], -mva_net['MVA'], 0.0]
bv = SWAP_NOTIONAL  # baseline: ATM swap has zero risk-free MtM

running = bv
bottoms = []
heights = []
colors_wf = []
for v in values[1:-1]:
    bottoms.append(min(running, running + v))
    heights.append(abs(v))
    colors_wf.append('firebrick' if v < 0 else 'steelblue')
    running += v

xva_bar_x = np.arange(1, len(values)-1)
ax.bar([0], [bv/1e4], width=0.5, color='gray', alpha=0.6)
ax.bar(xva_bar_x, [h/1e4 for h in heights], bottom=[b/1e4 for b in bottoms],
       width=0.5, color=colors_wf, alpha=0.8)
ax.bar([len(values)-1], [running/1e4], width=0.5, color='black', alpha=0.7)
ax.axhline(running/1e4, color='black', lw=0.8, ls='--', alpha=0.4)
ax.set_xticks(range(len(items)))
ax.set_xticklabels(items, fontsize=8)
ax.set_ylabel('Notional-normalised value (EUR × 10⁴ per 1M notional)')
ax.set_title(f'XVA waterfall — netting set\nFair value = {running/1e4:.2f}  '
             f'(bps adj = {(bv-running)/SWAP_NOTIONAL*10000:.2f})')

plt.tight_layout()
plt.show()


---
## 3. XVA Sensitivities — CS01, IR01, and Vol Sensitivity

### 3.1 Key Greeks for the XVA Desk

The XVA desk hedges the reserve in real time. The primary hedges are:

| Sensitivity | Description | Hedge instrument |
|---|---|---|
| **CS01** | CVA change per +1 bps counterparty CDS spread | CDS protection |
| **DVA01** | DVA change per +1 bps own CDS spread | Own bonds / CDS |
| **IR01** | FVA/MVA change per +1 bps parallel rate shift | IRS/swaption |
| **Vega** | CVA/FVA change per +1% IV shift (σ) | Swaption straddle |

### 3.2 CS01 from the CVA Model

$$\text{CS01} = \text{CVA}(S^{\text{CDS}} + 1\text{ bps}) - \text{CVA}(S^{\text{CDS}})$$

Under the flat hazard rate model, CS01 ≈ CVA × $\Delta t_{\text{avg}} / S^{\text{CDS}}$ for small spreads — approximately **linear in CVA** and inversely proportional to the spread.

### 3.3 IR01 for FVA and MVA

A +1 bps parallel shift in rates changes:
1. The IRS MTM → changes EE and NEE profiles → changes FVA
2. The DV01 of the swap → changes IM → changes MVA

For a payer IRS, a rate rise increases MTM → higher EE → higher FVA cost. For a receiver IRS (the hedge), the opposite applies — partially offsetting FVA IR01.


In [ ]:
# ── XVA sensitivity analysis ─────────────────────────────────────────────────

print("=== XVA Sensitivity Dashboard ===\n")

# 1. CS01 — CVA sensitivity to counterparty CDS spread
cva_up  = compute_cva(profile_net, CDS_CPTY + 1, RECOVERY)['CVA']
cs01    = cva_up - cva_net['CVA']
dv01_own_up = compute_dva(profile_net, nee_disc_net, CDS_OWN + 1, OWN_RECOV, sim, exp_dates)
dva01   = dv01_own_up['DVA'] - dva_net['DVA']

print("1. Credit sensitivities (1 bps bump):")
print(f"   CS01 (counterparty)  = {cs01:+,.2f} EUR  ({cs01/SWAP_NOTIONAL*10000:+.4f} bps/notional)")
print(f"   DVA01 (own)          = {dva01:+,.2f} EUR  ({dva01/SWAP_NOTIONAL*10000:+.4f} bps/notional)")

# 2. CVA vs CDS spread curve
print("\n2. CVA vs counterparty CDS spread:")
for cds in [30, 60, 100, 150, 250]:
    cva_c = compute_cva(profile_net, cds, RECOVERY)['CVA']
    print(f"   CDS={cds:>4.0f}bps  CVA={cva_c:>10,.2f} EUR  ({cva_c/SWAP_NOTIONAL*10000:.2f}bps)")

# 3. FVA sensitivity to funding spread
print("\n3. FVA vs funding spread (no CSA, symmetric s_f=s_r):")
for sf in [10, 20, 30, 50, 75]:
    fva_s = compute_fva(profile_net['EE_disc'], nee_disc_net, exp_dates, sf, sf, 'none')
    print(f"   s_f={sf:>3.0f}bps  FVA={fva_s['FVA']:>10,.2f} EUR  "
          f"(FCA={fva_s['FCA']:>8,.2f}, FBA={fva_s['FBA']:>8,.2f})")

# 4. MVA: bilateral vs cleared for the netting set
print("\n4. MVA: bilateral vs cleared for the netting set:")
for mpor in [5, 10]:
    mva_m = compute_mva_from_dv01(
        sim.paths, exp_dates, sim,
        pd_list, yf_list, no_list,
        KAPPA, SIGMA, ois, SIMM_RW_BPS, mpor, IM_FUND_SPREAD
    )
    label = 'Bilateral' if mpor==10 else 'Cleared  '
    print(f"   {label} (MPOR={mpor}d)  MVA={mva_m['MVA']:>10,.2f} EUR  "
          f"({mva_m['MVA']/SWAP_NOTIONAL*10000:.2f}bps)")

# 5. XVA breakdown vs CSA status
print("\n5. XVA under different CSA arrangements (CVA+FVA+MVA):")
for csa, label in [('two_way','Two-way CSA'), ('one_way_post','One-way (we post)'),
                    ('none','No CSA')]:
    fva_c = compute_fva(profile_net['EE_disc'], nee_disc_net, exp_dates,
                         S_FUND, S_INVEST, csa)
    mva_c = mva_net['MVA'] if csa == 'none' else mva_net['MVA'] * 0.5  # simplified
    total = cva_net['CVA'] + fva_c['FVA'] + mva_c
    print(f"   {label:<22}  CVA={cva_net['CVA']:>8,.0f}  "
          f"FVA={fva_c['FVA']:>8,.0f}  MVA≈{mva_c:>8,.0f}  "
          f"Total={total:>8,.0f} EUR")


---
## Summary

### Full XVA Framework

$$V^{\text{fair value}} = V^{\text{risk-free}} \underbrace{- \text{CVA}}_{\text{cpty default cost}} \underbrace{+ \text{DVA}}_{\text{own default benefit}} \underbrace{- \text{FVA}}_{\text{funding cost}} \underbrace{- \text{MVA}}_{\text{IM funding cost}}$$

### Netting Benefit Summary

| Component | Without netting | With netting | Reduction |
|---|---|---|---|
| CVA | sum of trade CVAs | portfolio CVA on net EE | Depends on exposure correlation |
| FVA | sum of trade FVAs | portfolio FVA on net EE/NEE | Offsets when EE and NEE cancel |
| MVA | sum of trade IM × s_IM | net DV01 × RW × s_IM | **Largest when trades partially hedge** |

### Key Design Principles Demonstrated

Every function in this series takes all trade and credit parameters as explicit arguments:

| Function | Arguments (all explicit) |
|---|---|
| `irs_mtm()` | paths, t, payment_dates, year_fracs, K, notional, is_payer, κ, σ, curve, dt, n_steps |
| `compute_cva()` | profile, cds_bps, recovery |
| `compute_dva()` | profile, nee_disc, own_cds_bps, own_recovery, sim, dates |
| `compute_fva()` | ee_disc, nee_disc, dates, s_f_bps, s_r_bps, csa_type |
| `compute_mva_from_dv01()` | paths, dates, sim, payment_dates_list, year_fracs_list, notionals, κ, σ, curve, rw_bps, mpor_days, s_im_bps |

Adding a new trade to the netting set is a single list append. Changing any credit/funding parameter is a single argument change. No global state anywhere.

**Next:** [QRE-57 — XVA OOP Class](12_xva_class.ipynb) — wraps this computation chain into a production `XVAEngine` class that accepts a netting set and computes the full CVA/DVA/FVA/MVA in one call, with built-in sensitivity computation and SA-CVA scaffolding.
